# 🏸 Badminton Match Outcome Prediction via Dynamic ELO

**Pipeline:**
1. Data Loading & Cleaning
2. Dynamic ELO Rating System (paper-based, with tunable hyperparameters)
3. Feature Engineering (recent form, head-to-head, rest days)
4. Hyperparameter Optimisation (grid search with time-series CV)
5. Model Training & Evaluation (XGBoost / HGB)
6. EDA Visualisations & Rating Leaderboard


In [ ]:
# ── Standard library & third-party ──────────────────────────────────────────
import os, re, ast, warnings, math
from collections import defaultdict, deque
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pandas.plotting import scatter_matrix
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.metrics import (
    log_loss, accuracy_score, brier_score_loss,
    roc_auc_score, RocCurveDisplay, CalibrationDisplay
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Try XGBoost; fall back to HGB silently
try:
    from xgboost import XGBClassifier
    USE_XGB = True
except ImportError:
    USE_XGB = False
    print('XGBoost not found – using HistGradientBoostingClassifier.')

print('All imports OK. XGBoost:', USE_XGB)

## 1 · Data Loading

In [ ]:
# ── Helper: parse player name from a cell ────────────────────────────────────
def _first_name(s):
    """Return the first player name from a (possibly list-formatted) cell."""
    if pd.isna(s):
        return None
    if isinstance(s, (list, tuple)):
        return str(s[0]).strip() if s else None
    try:
        val = ast.literal_eval(str(s))
        if isinstance(val, (list, tuple)) and val:
            return str(val[0]).strip()
    except Exception:
        pass
    raw = str(s).strip()
    if ';' in raw:
        return raw.split(';')[0].strip()
    return raw


def _build_name_normalizer(series_list):
    """
    Build a name→canonical mapping to merge spelling variants of the same player.

    Strategy: collapse spaces and hyphens to get a grouping key.
    e.g. 'Yu Fei Chen' and 'Yufei Chen' both → 'yufeichen' → same canonical name.
    The most-frequently-seen variant becomes the canonical display name.
    """
    from collections import Counter as _Counter
    groups = defaultdict(list)
    for s in series_list:
        for name in s.dropna():
            key = re.sub(r'[\s\-]+', '', str(name).strip().lower())
            groups[key].append(str(name).strip())

    mapping = {}
    merged = 0
    for key, variants in groups.items():
        canonical = _Counter(variants).most_common(1)[0][0]
        unique_variants = set(variants)
        if len(unique_variants) > 1:
            merged += 1
            print(f'  Name merge: {sorted(unique_variants)} → "{canonical}"')
        for v in unique_variants:
            mapping[v] = canonical
    print(f'Name normaliser built: {len(mapping)} entries, {merged} variant groups merged.')
    return mapping


def load_singles(ms_csv='ms.csv', ws_csv='ws.csv', combined_csv='singles_all.csv'):
    """
    Load and combine MS/WS singles data.  Accepts either a pre-merged file or
    separate ms/ws CSVs, with automatic player-column detection.
    """
    if os.path.exists(combined_csv):
        df = pd.read_csv(combined_csv, low_memory=False)
        print(f'Loaded {combined_csv}: {df.shape}')
    else:
        frames = []
        for path, disc in [(ms_csv, 'MS'), (ws_csv, 'WS')]:
            if not os.path.exists(path):
                raise FileNotFoundError(path)
            tmp = pd.read_csv(path, low_memory=False)
            tmp['discipline'] = disc
            frames.append(tmp)
            print(f'Loaded {path}: {tmp.shape}')
        all_cols = sorted(set().union(*(f.columns for f in frames)))
        df = pd.concat(
            [f.reindex(columns=all_cols) for f in frames],
            ignore_index=True
        )
        print(f'Combined shape: {df.shape}')

    # ── Normalise key columns ────────────────────────────────────────────────
    if 'discipline' in df.columns:
        df['discipline'] = df['discipline'].astype(str).str.strip().str.upper()

    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # player_A / player_B  (reuse if present, otherwise parse team columns)
    if 'player_A' not in df.columns or 'player_B' not in df.columns:
        df['player_A'] = df['team_one_players'].apply(_first_name)
        df['player_B'] = df['team_two_players'].apply(_first_name)

    # Canonical integer winner flag
    if 'winner' not in df.columns and 'target_A_win' in df.columns:
        df['winner'] = df['target_A_win'].apply(lambda x: 1 if x == 1 else 2)

    df['winner'] = pd.to_numeric(df['winner'], errors='coerce')

    # ── Normalise player names to merge spelling variants ────────────────────
    # e.g. 'Yu Fei Chen' and 'Yufei Chen' are the same athlete → unified here
    print('Building player name normaliser…')
    _name_map = _build_name_normalizer([df['player_A'], df['player_B']])
    df['player_A'] = df['player_A'].map(lambda x: _name_map.get(str(x).strip(), x) if pd.notna(x) else x)
    df['player_B'] = df['player_B'].map(lambda x: _name_map.get(str(x).strip(), x) if pd.notna(x) else x)

    # Drop rows without essential data
    before = len(df)
    df = df.dropna(subset=['player_A', 'player_B', 'winner', 'date']).reset_index(drop=True)
    print(f'After cleaning: {len(df)} rows ({before - len(df)} dropped)')

    # Sort chronologically (critical for ELO + feature engineering)
    df = df.sort_values('date').reset_index(drop=True)
    return df


df = load_singles()
print('\nDate range:', df['date'].min().date(), '→', df['date'].max().date())
print('Disciplines:', df['discipline'].value_counts().to_dict() if 'discipline' in df.columns else 'n/a')
print('Unique players:', len(set(df['player_A'].tolist() + df['player_B'].tolist())))

## 2 · Dynamic ELO System

In [ ]:
# ── Score fraction for decisive scorelines (BWF-paper convention) ─────────────
_S = {'2-0': 21/22, '2-1': 21/32, '0-2': 1/22, '1-2': 11/32}


def _round_to_K(round_str, K_scale=1.0):
    """Map round name → base K value, then scale it."""
    if pd.isna(round_str):
        return 80 * K_scale
    r = re.sub(r'\s+', ' ', str(round_str).lower().strip())
    if 'qualification' in r and 'quarter' in r: base = 100
    elif 'qualification' in r and ('round of 16' in r or '16' in r): base = 90
    elif 'final' in r and 'qualification' not in r: base = 190
    elif 'semi' in r: base = 170
    elif 'quarter' in r and 'qualification' not in r: base = 140
    elif 'round of 16' in r or re.search(r'\b16\b', r): base = 110
    elif 'round of 32' in r or re.search(r'\b32\b', r): base = 80
    elif re.search(r'\bround\s*1\b', r): base = 50
    elif re.search(r'\bround\s*2\b', r): base = 60
    elif re.search(r'\bround\s*3\b', r): base = 70
    else: base = 80
    return base * K_scale


def _parse_score_str(s):
    """Parse 'game_1_score, game_2_score, …' into decisive label."""
    if pd.isna(s):
        return None
    a_wins = b_wins = 0
    for part in re.split(r'[;,]\s*', str(s)):
        m = re.findall(r'(\d+)\s*[-:]\s*(\d+)', part)
        if m:
            a, b = int(m[0][0]), int(m[0][1])
            if a > b: a_wins += 1
            elif b > a: b_wins += 1
    if a_wins + b_wins == 0:
        return None
    key = f'{a_wins}-{b_wins}'
    return key if key in _S else None


def compute_elo(
    df,
    *,
    K_scale=1.0,
    streak_C=20.0,
    streak_min=2,
    max_delta=200.0,
    init_rating=1500.0,
    xi=400.0,
    playerA_col='player_A',
    playerB_col='player_B',
    winner_col='winner',
    date_col='date',
    round_col='round',
):
    """
    Compute paper-based dynamic ELO ratings.

    Improvements over the original:
    • Single canonical function (no duplication)
    • K_scale parameter exposes scaling without redefining K table
    • Decisive score fraction is looked up before per-game parsing
    • Returns a copy; original DataFrame is never mutated
    """
    out = df.copy().sort_values(date_col).reset_index(drop=True)

    score_cols = [c for c in ['game_1_score', 'game_2_score', 'game_3_score'] if c in out.columns]

    ratings  = defaultdict(lambda: init_rating)
    streaks  = defaultdict(int)

    Ra_before, Rb_before, Ra_after, Rb_after = [], [], [], []

    for _, row in out.iterrows():
        a = row[playerA_col]
        b = row[playerB_col]

        Ra = ratings[a]
        Rb = ratings[b]
        Ra_before.append(Ra)
        Rb_before.append(Rb)

        w      = row[winner_col]
        A_won  = (w == 1) or (str(w).strip() == '1')
        B_won  = (w == 2) or (str(w).strip() == '2')

        # Score fraction
        combined = ','.join(str(row.get(c, '')) for c in score_cols)
        decisive = _parse_score_str(combined)
        if decisive:
            s_A = _S[decisive]
        else:
            s_A = 1.0 if A_won else (0.0 if B_won else 0.5)
        s_B = 1.0 - s_A

        P_A = 1.0 / (1.0 + 10 ** ((Rb - Ra) / xi))
        P_B = 1.0 - P_A
        K   = _round_to_K(row.get(round_col), K_scale)

        # Walkover → halve the K impact
        is_walkover = any(
            (c in out.columns) and bool(row.get(c, False))
            for c in ('retired', 'walkover', 'walk_over')
        )
        if is_walkover:
            K *= 0.5

        # Streak bonus for the winning side
        if A_won:
            C = streak_C if streaks[a] >= streak_min else 0.0
            dA = K * (s_A - P_A) + C
            dB = K * (s_B - P_B)
            streaks[a] += 1; streaks[b] = 0
        elif B_won:
            C = streak_C if streaks[b] >= streak_min else 0.0
            dB = K * (s_B - P_B) + C
            dA = K * (s_A - P_A)
            streaks[b] += 1; streaks[a] = 0
        else:
            dA = dB = 0.0

        # Clip deltas
        dA = max(min(dA, max_delta), -max_delta)
        dB = max(min(dB, max_delta), -max_delta)

        ratings[a] += dA
        ratings[b] += dB
        Ra_after.append(ratings[a])
        Rb_after.append(ratings[b])

    out['elo_A_before']  = Ra_before
    out['elo_B_before']  = Rb_before
    out['elo_A_after']   = Ra_after
    out['elo_B_after']   = Rb_after
    out['elo_diff']      = out['elo_A_before'] - out['elo_B_before']
    return out, dict(ratings)


# Default run with conservative parameters
df_elo, final_ratings = compute_elo(df, K_scale=0.6, streak_C=0, max_delta=100)

print('ELO range (A_before):', df_elo['elo_A_before'].min().round(1),
      '→', df_elo['elo_A_before'].max().round(1))
print('ELO range (A_after): ', df_elo['elo_A_after'].min().round(1),
      '→', df_elo['elo_A_after'].max().round(1))

## 3 · Feature Engineering

In [ ]:
def add_features(df_in, window=5, decay=1.5):
    """
    Adds time-aware, no-leakage features:
      • elo_diff              – already computed by compute_elo
      • weighted_form_diff    – exponentially-decayed recent win rate diff
      • h2h_diff              – prior head-to-head win difference
      • rest_days_diff        – days since last match (A minus B)
    All features use ONLY information available BEFORE each match.
    """
    df = df_in.copy().sort_values('date').reset_index(drop=True)

    # Weight vector (most recent = highest weight)
    weights = np.exp(-np.linspace(0, decay, window))
    weights /= weights.sum()

    last_results = defaultdict(lambda: deque(maxlen=window))
    last_date    = {}
    h2h          = defaultdict(lambda: defaultdict(int))

    wf_A, wf_B, h2h_diff, rest_A_list, rest_B_list = [], [], [], [], []

    for _, row in df.iterrows():
        a, b, d = row['player_A'], row['player_B'], row['date']

        # ─ Weighted form ─────────────────────────────────────────────────
        for player, lst in [(a, wf_A), (b, wf_B)]:
            arr = np.array(last_results[player]) if last_results[player] else np.array([])
            if len(arr) < window:
                arr = np.concatenate([np.zeros(window - len(arr)), arr])
            lst.append(float((weights * arr).sum()))

        # ─ Head-to-head (Bayesian-shrunk) ───────────────────────────────
        # Raw h2h_diff (e.g. 10-0 → 10) dominated feature importance at ~0.73.
        # Replace with a Bayesian win-rate difference bounded in (-0.5, +0.5):
        #   0.0  when they've never met  (prior pulls to 0.5 each)
        #   ±0.45 max even after 10 dominant wins (avoids extreme scale effects)
        _n_ab = h2h[a][b]
        _n_ba = h2h[b][a]
        _total = _n_ab + _n_ba
        _H2H_PRIOR = 1.0          # 0.5 virtual win each side before first meeting
        _p_A = (_n_ab + _H2H_PRIOR / 2) / (_total + _H2H_PRIOR)
        h2h_diff.append(_p_A - 0.5)  # centred; range (-0.5, +0.5)

        # ─ Rest days ─────────────────────────────────────────────────────
        for player, lst in [(a, rest_A_list), (b, rest_B_list)]:
            if player in last_date and pd.notna(d) and pd.notna(last_date[player]):
                lst.append((d - last_date[player]).days)
            else:
                lst.append(30.0)   # cold-start default

        # ─ Update state (AFTER features are recorded) ────────────────────
        w = row['winner']
        A_won = (w == 1) or (str(w).strip() == '1')
        if A_won:
            last_results[a].append(1); last_results[b].append(0)
            h2h[a][b] += 1
        else:
            last_results[a].append(0); last_results[b].append(1)
            h2h[b][a] += 1
        last_date[a] = d
        last_date[b] = d

    df['wform_A']        = wf_A
    df['wform_B']        = wf_B
    df['wform_diff']     = df['wform_A'] - df['wform_B']
    df['h2h_diff']       = h2h_diff
    df['rest_A']         = rest_A_list
    df['rest_B']         = rest_B_list
    df['rest_diff']      = df['rest_A'] - df['rest_B']

    # Null-safe fill for rank/seed if present
    # BUG FIX: fillna(0) treated unranked players as rank-0 (better than #1).
    # Use 999 as a sentinel so unranked players appear as very weak, not very strong.
    # ⚠ LEAKAGE CHECK: ensure rank_A/rank_B are BWF rankings BEFORE the match,
    #   not year-end / post-tournament rankings – those would leak future information.
    _RANK_SENTINEL = 999
    for ca, cb, name in [('rank_A','rank_B','rank_diff'),
                          ('seed_A','seed_B','seed_diff')]:
        if ca in df.columns and cb in df.columns:
            df[name] = (pd.to_numeric(df[ca], errors='coerce').fillna(_RANK_SENTINEL)
                      - pd.to_numeric(df[cb], errors='coerce').fillna(_RANK_SENTINEL))
        else:
            df[name] = 0.0

    df['target'] = (df['winner'] == 1).astype(int)
    return df


df_feat = add_features(df_elo)

FEATURES_ELO = ['elo_diff']
FEATURES_ALL = ['elo_diff', 'wform_diff', 'h2h_diff', 'rest_diff',
                 'rank_diff', 'seed_diff']

print('Feature sample:')
df_feat[FEATURES_ALL + ['target']].describe().round(2)


def get_temporal_splits(df_in, train_frac=0.70, val_frac=0.15):
    """
    Choose chronological train/validation/test cut points from the available dates.

    Splits are based on unique dates so that the same calendar day never appears
    in more than one partition.
    """
    dates = (
        pd.Series(pd.to_datetime(df_in['date'], errors='coerce').dropna().unique())
        .sort_values()
        .reset_index(drop=True)
    )
    if len(dates) < 30:
        raise ValueError("Not enough unique dates to create temporal splits.")

    train_end_idx = max(1, int(len(dates) * train_frac))
    test_start_idx = max(train_end_idx + 1, int(len(dates) * (train_frac + val_frac)))
    test_start_idx = min(test_start_idx, len(dates) - 1)

    val_start = dates.iloc[train_end_idx]
    test_start = dates.iloc[test_start_idx]
    if test_start <= val_start:
        test_start = dates.iloc[min(train_end_idx + 1, len(dates) - 1)]

    return val_start, test_start


## 4 · ELO Hyperparameter Optimisation

In [ ]:

def evaluate_params(df_raw, K_scale, streak_C, max_delta,
                    train_frac=0.70, val_frac=0.15):
    """
    Recompute ELO with given params, add features, then score them on a
    chronological validation slice.

    The validation slice is never used to update ELO/form/H2H state, and the
    final test slice is left untouched here.
    """
    val_start, test_start = get_temporal_splits(df_raw, train_frac=train_frac, val_frac=val_frac)

    df_e, _ = compute_elo(df_raw, K_scale=K_scale, streak_C=streak_C,
                          max_delta=max_delta)
    df_f    = add_features(df_e)
    df_m    = df_f.dropna(subset=['elo_diff', 'target']).reset_index(drop=True)

    train   = df_m[df_m['date'] < val_start].copy()
    val     = df_m[(df_m['date'] >= val_start) & (df_m['date'] < test_start)].copy()

    if len(train) < 50 or len(val) < 30:
        return np.nan, 0

    # FIX: use the same full feature set as the final model so that the chosen
    # ELO hyperparameters are co-optimal with rank/form/h2h features, not just ELO alone.
    # Fall back to FEATURES_ELO if rank/seed columns are absent in this split.
    available = [f for f in FEATURES_ALL if f in train.columns and f in val.columns]
    feats_cv = available if len(available) > 1 else FEATURES_ELO

    X_tr = train[feats_cv].fillna(0)
    X_val = val[feats_cv].fillna(0)
    y_tr = train['target'].values
    y_val = val['target'].values

    m = HGB(random_state=42)
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_val)[:, 1]
    return log_loss(y_val, p), len(val)


param_grid = {
    'K_scale':   [0.4, 0.6, 0.8, 1.0],
    'streak_C':  [0, 10, 20],
    'max_delta': [80, 100, 150, 200],
}

results = []
best_loss = np.inf
best_params = None

combos = list(product(param_grid['K_scale'],
                      param_grid['streak_C'],
                      param_grid['max_delta']))

print(f'Running {len(combos)} parameter combinations…')
for ks, sc, md in combos:
    ll, n = evaluate_params(df, K_scale=ks, streak_C=sc, max_delta=md)
    results.append({'K_scale': ks, 'streak_C': sc, 'max_delta': md,
                    'logloss': ll, 'n_val': n})
    if ll < best_loss:
        best_loss   = ll
        best_params = {'K_scale': ks, 'streak_C': sc, 'max_delta': md}

grid_df = pd.DataFrame(results).sort_values('logloss').reset_index(drop=True)
print('\\n── Top 10 configurations (validation) ─────────────────')
print(grid_df.head(10).to_string(index=False))
print('\\nBest params:', best_params, f'  val_logloss={best_loss:.5f}')


In [ ]:

# ── Re-compute ELO with the best parameters ───────────────────────────────────
df_best, final_ratings_best = compute_elo(df, **best_params)
df_final = add_features(df_best)
df_final['target'] = (df_final['winner'] == 1).astype(int)

VAL_START, TEST_START = get_temporal_splits(df_final)

print('Final dataset shape:', df_final.shape)
print('Validation start:', VAL_START.date())
print('Test start:', TEST_START.date())
print('ELO range after best params:',
      df_final['elo_A_after'].min().round(1), '→',
      df_final['elo_A_after'].max().round(1))


## 5 · Model Training & Evaluation

In [ ]:

train_base = df_final[df_final['date'] < VAL_START].copy()
val        = df_final[(df_final['date'] >= VAL_START) & (df_final['date'] < TEST_START)].copy()
test       = df_final[df_final['date'] >= TEST_START].copy()

print(f'Train base: {len(train_base)}  |  Val: {len(val)}  |  Test: {len(test)}')

y_train_base = train_base['target'].values
y_val        = val['target'].values
y_test       = test['target'].values


def make_clf():
    if USE_XGB:
        return XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, n_jobs=-1
        )
    return HGB(max_iter=300, max_depth=4, learning_rate=0.05,
               random_state=42)


feature_sets = {
    'ELO only':         ['elo_diff'],
    'ELO + form':       ['elo_diff', 'wform_diff'],
    'ELO + form + h2h': ['elo_diff', 'wform_diff', 'h2h_diff'],
    'All features':     FEATURES_ALL,
}

def _fit_score_on_slice(train_df, eval_df, feats):
    X_tr = train_df[feats].fillna(0)
    X_ev = eval_df[feats].fillna(0)
    y_tr = train_df['target'].values
    y_ev = eval_df['target'].values

    base = make_clf()
    base.fit(X_tr, y_tr)

    n_splits = 5 if len(train_df) >= 500 else 3
    cal = CalibratedClassifierCV(
        estimator=make_clf(),
        cv=TimeSeriesSplit(n_splits=n_splits),
        method='sigmoid'
    )
    cal.fit(X_tr, y_tr)

    p_raw = base.predict_proba(X_ev)[:, 1]
    p_cal = cal.predict_proba(X_ev)[:, 1]

    return base, cal, p_raw, p_cal


# Step 1: choose the feature set on the validation period only
val_rows = []
val_models = {}

for name, feats in feature_sets.items():
    base, cal, p_raw, p_cal = _fit_score_on_slice(train_base, val, feats)

    row = {
        'Feature set':  name,
        'n_features':   len(feats),
        'Val LogLoss (raw)': round(log_loss(y_val, p_raw), 4),
        'Val LogLoss (cal)': round(log_loss(y_val, p_cal), 4),
        'Val Accuracy':     round(accuracy_score(y_val, p_raw > 0.5), 4),
        'Val Brier':        round(brier_score_loss(y_val, p_raw), 4),
        'Val AUC-ROC':      round(roc_auc_score(y_val, p_raw), 4),
    }
    val_rows.append(row)
    val_models[name] = (base, cal, feats, p_raw, p_cal)

val_eval_df = pd.DataFrame(val_rows)
print('\\n── Model comparison (validation set) ─────────────────────')
print(val_eval_df.to_string(index=False))

best_feat_name = val_eval_df.sort_values('Val LogLoss (raw)').iloc[0]['Feature set']
best_base, best_cal, best_feats, val_p_raw, val_p_cal = val_models[best_feat_name]
print(f'\\nBest feature set from validation: {best_feat_name}  {best_feats}')

# Step 2: refit the winning feature set on train+validation and score once on test
train_plus_val = pd.concat([train_base, val]).sort_values('date').reset_index(drop=True)

X_fit = train_plus_val[best_feats].fillna(0)
y_fit = train_plus_val['target'].values
X_test = test[best_feats].fillna(0)

final_clf = make_clf()
final_clf.fit(X_fit, y_fit)

final_cal = CalibratedClassifierCV(
    estimator=make_clf(),
    cv=TimeSeriesSplit(n_splits=5 if len(train_plus_val) >= 500 else 3),
    method='sigmoid'
)
final_cal.fit(X_fit, y_fit)

test_p_raw = final_clf.predict_proba(X_test)[:, 1]
test_p_cal = final_cal.predict_proba(X_test)[:, 1]

test_eval_df = pd.DataFrame([{
    'Feature set': best_feat_name,
    'n_features': len(best_feats),
    'Test LogLoss (raw)': round(log_loss(y_test, test_p_raw), 4),
    'Test LogLoss (cal)': round(log_loss(y_test, test_p_cal), 4),
    'Test Accuracy': round(accuracy_score(y_test, test_p_raw > 0.5), 4),
    'Test Brier': round(brier_score_loss(y_test, test_p_raw), 4),
    'Test AUC-ROC': round(roc_auc_score(y_test, test_p_raw), 4),
}])

print('\\n── Final untouched test-set performance ──────────────────')
print(test_eval_df.to_string(index=False))

# ── AUC sanity: feature ablation on test set ─────────────────────────────────
# High AUC (>0.90) often indicates strong features (rank/seed) rather than leakage.
# This ablation shows how much each feature group contributes.
print('\\n── AUC ablation (test set) – diagnosing the high AUC ────────')
ablation_rows = []
for ablation_feats, label in [
    (['elo_diff'],                           'ELO only'),
    (['elo_diff', 'wform_diff'],             'ELO + form'),
    (['elo_diff', 'wform_diff', 'h2h_diff'], 'ELO + form + h2h'),
    (['rank_diff'],                          'rank_diff only'),
    (['seed_diff'],                          'seed_diff only'),
    (FEATURES_ALL,                           'All features'),
]:
    avail = [f for f in ablation_feats if f in test.columns]
    if not avail:
        continue
    _clf = make_clf()
    _clf.fit(train_plus_val[avail].fillna(0), y_fit)
    _p = _clf.predict_proba(test[avail].fillna(0))[:, 1]
    ablation_rows.append({'Features': label,
                          'Test AUC': round(roc_auc_score(y_test, _p), 4),
                          'Test Acc': round(accuracy_score(y_test, _p > 0.5), 4)})
ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))
print('\\n⚠  If rank_diff only / seed_diff only already gives AUC > 0.90,')
print('   verify those columns are PRE-MATCH rankings, not post-tournament ones.')

# Keep convenient handles for later cells
models = {
    best_feat_name: (final_clf, final_cal, best_feats, test_p_raw, test_p_cal)
}
eval_df = val_eval_df


In [ ]:

# ── Time-series cross-validation on the selected feature set ───────────────
best_feat_name = val_eval_df.sort_values('Val LogLoss (raw)').iloc[0]['Feature set']
_, _, best_feats, _, _ = models[best_feat_name]
print(f'Best feature set: {best_feat_name}  {best_feats}')

df_cv = train_plus_val.dropna(subset=best_feats + ['target']).reset_index(drop=True)
X_cv  = df_cv[best_feats].fillna(0).values
y_cv  = df_cv['target'].values

tscv = TimeSeriesSplit(n_splits=5)
cv_losses, cv_accs, cv_aucs = [], [], []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_cv), 1):
    m = make_clf()
    m.fit(X_cv[tr_idx], y_cv[tr_idx])
    p = m.predict_proba(X_cv[va_idx])[:, 1]
    cv_losses.append(log_loss(y_cv[va_idx], p))
    cv_accs.append(accuracy_score(y_cv[va_idx], p > 0.5))
    cv_aucs.append(roc_auc_score(y_cv[va_idx], p))
    print(f'  Fold {fold}: logloss={cv_losses[-1]:.4f}  acc={cv_accs[-1]:.4f}  auc={cv_aucs[-1]:.4f}')

print(f'\\nCV mean  → logloss={np.mean(cv_losses):.4f}  acc={np.mean(cv_accs):.4f}  auc={np.mean(cv_aucs):.4f}')
print(f'CV stdev → logloss={np.std(cv_losses):.4f}  acc={np.std(cv_accs):.4f}  auc={np.std(cv_aucs):.4f}')


## 6 · Visualisations

In [ ]:

# ── Feature importances + ROC + Calibration ───────────────────────────────────
best_clf, best_cal, best_feats, p_raw, p_cal = models[best_feat_name]
X_te_best = test[best_feats].fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ─ (A) Feature importances ──────────────────────────────────────────────────
try:
    imp = best_clf.feature_importances_
    feat_ser = pd.Series(imp, index=best_feats).sort_values()
    feat_ser.plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('Feature Importances', fontweight='bold')
    axes[0].set_xlabel('Importance score')
except Exception:
    axes[0].text(0.5, 0.5, 'Feature importances\nnot available',
                 ha='center', va='center', transform=axes[0].transAxes)

# ─ (B) ROC curve ────────────────────────────────────────────────────────────
RocCurveDisplay.from_predictions(y_test, p_raw, name=best_feat_name, ax=axes[1])
axes[1].set_title('ROC Curve (untouched test set)', fontweight='bold')
axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8)

# ─ (C) Calibration curve ────────────────────────────────────────────────────
CalibrationDisplay.from_predictions(
    y_test, p_raw, n_bins=10, name='Uncalibrated', ax=axes[2])
CalibrationDisplay.from_predictions(
    y_test, p_cal, n_bins=10, name='Calibrated (Platt)', ax=axes[2],
    color='tomato')
axes[2].set_title('Calibration Curve (untouched test set)', fontweight='bold')

plt.suptitle(f'Final Model Evaluation — {best_feat_name}', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight')
plt.show()
print('Saved model_evaluation.png')


In [ ]:
# ── ELO distribution & pairplot ───────────────────────────────────────────────
# Final rating leaderboard
ratings_df = (
    pd.DataFrame.from_dict(final_ratings_best, orient='index', columns=['elo'])
    .reset_index()
    .rename(columns={'index': 'player'})
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
# Merge discipline if possible
if 'discipline' in df_final.columns:
    disc_map = {}
    for _, row in df_final.iterrows():
        disc_map.setdefault(row['player_A'], row.get('discipline'))
        disc_map.setdefault(row['player_B'], row.get('discipline'))
    ratings_df['discipline'] = ratings_df['player'].map(disc_map)

print('Top 20 final ELO ratings:')
print(ratings_df.head(20).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ─ ELO distribution ─────────────────────────────────────────────────────────
axes[0].hist(ratings_df['elo'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(1500, color='red', linestyle='--', label='Init (1500)')
axes[0].set_title('Final ELO Distribution', fontweight='bold')
axes[0].set_xlabel('ELO rating')
axes[0].set_ylabel('# players')
axes[0].legend()

# ─ Top-15 bar chart ─────────────────────────────────────────────────────────
top15 = ratings_df.head(15)
colors = ['#2196F3' if d == 'MS' else '#E91E63' if d == 'WS' else '#4CAF50'
          for d in top15.get('discipline', ['MS'] * 15)]
axes[1].barh(top15['player'][::-1], top15['elo'][::-1], color=colors[::-1])
axes[1].set_title('Top 15 Players by ELO', fontweight='bold')
axes[1].set_xlabel('ELO rating')
# legend
from matplotlib.patches import Patch
handles = [Patch(color='#2196F3', label='MS'), Patch(color='#E91E63', label='WS')]
axes[1].legend(handles=handles)

plt.tight_layout()
plt.savefig('elo_ratings.png', bbox_inches='tight')
plt.show()
print('Saved elo_ratings.png')

In [ ]:
# ── Pairplot (improved) ───────────────────────────────────────────────────────
plot_cols = ['elo_diff', 'wform_diff', 'h2h_diff', 'rest_diff']
sample_df = df_final[plot_cols + ['target']].dropna().sample(
    min(2000, len(df_final)), random_state=42)
sample_df['Result'] = sample_df['target'].map({1: 'A wins', 0: 'B wins'})

g = sns.pairplot(
    sample_df,
    vars=plot_cols,
    hue='Result',
    palette={'A wins': '#2196F3', 'B wins': '#E91E63'},
    diag_kind='kde',
    plot_kws={'alpha': 0.35, 's': 12},
    diag_kws={'linewidth': 1.5},
    corner=True,
)
g.fig.suptitle('Pairwise Feature Distributions (coloured by outcome)',
               y=1.02, fontsize=12, fontweight='bold')
g.fig.set_size_inches(10, 8)
plt.tight_layout()
plt.savefig('pairplot.png', bbox_inches='tight')
plt.show()
print('Saved pairplot.png')

In [ ]:
# ── ELO progression for top players over time ─────────────────────────────────
TOP_N = 6
top_players = ratings_df.head(TOP_N)['player'].tolist()

fig, ax = plt.subplots(figsize=(13, 5))
palette = sns.color_palette('tab10', TOP_N)

for player, color in zip(top_players, palette):
    mask_A = df_final['player_A'] == player
    mask_B = df_final['player_B'] == player

    timeline = pd.concat([
        df_final.loc[mask_A, ['date', 'elo_A_after']].rename(columns={'elo_A_after': 'elo'}),
        df_final.loc[mask_B, ['date', 'elo_B_after']].rename(columns={'elo_B_after': 'elo'}),
    ]).sort_values('date')

    if len(timeline) > 1:
        ax.plot(timeline['date'], timeline['elo'], label=player, color=color, linewidth=1.6)

ax.axhline(1500, color='grey', linestyle='--', linewidth=0.8, label='Init')
ax.set_title(f'ELO Progression – Top {TOP_N} Players', fontweight='bold', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('ELO Rating')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('elo_progression.png', bbox_inches='tight')
plt.show()
print('Saved elo_progression.png')

In [ ]:
# ── Grid search heat-map ──────────────────────────────────────────────────────
pivot = grid_df.pivot_table(
    index='K_scale', columns='max_delta', values='logloss', aggfunc='min')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn_r', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Min log-loss'})
ax.set_title('Grid Search: min log-loss by K_scale × max_delta', fontweight='bold')
plt.tight_layout()
plt.savefig('grid_search_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved grid_search_heatmap.png')

## 7 · Export Artefacts

In [ ]:

# ── Save key outputs ──────────────────────────────────────────────────────────
ratings_df.to_csv('final_elo_ratings.csv', index=False)
print('Saved final_elo_ratings.csv')

df_final[['date', 'player_A', 'player_B', 'winner', 'elo_diff',
          'wform_diff', 'h2h_diff', 'rest_diff', 'target']]\
    .to_csv('model_dataset.csv', index=False)
print('Saved model_dataset.csv')

grid_df.to_csv('elo_grid_search_results.csv', index=False)
print('Saved elo_grid_search_results.csv')

val_eval_df.to_csv('validation_model_comparison.csv', index=False)
print('Saved validation_model_comparison.csv')

test_eval_df.to_csv('final_test_performance.csv', index=False)
print('Saved final_test_performance.csv')

print('\n── Final summary ────────────────────────────────────────')
print('Validation comparison:')
print(val_eval_df.to_string(index=False))
print('\nFinal untouched test performance:')
print(test_eval_df.to_string(index=False))
